# Tabular closure model (role B, task 8)

Predicts whether an active restaurant closes in the 12 months after the cutoff, from
pre-cutoff review statistics, location, chain size and categories.
Feature code: `src/tabular.py`. Labels and split: `src/labels.py`, unchanged.

Train = 2017-12-31 snapshot (2018 closures), test = 2018-12-31 snapshot (2019 closures).
Each model is checked on both labels: `"closed"` (default) and `"exclude"`.

Scoring here is a single average-precision check while building the model. The official
comparison will use the shared evaluation function (task 6) once it is in `src/`.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.inspection import permutation_importance
from sklearn.metrics import average_precision_score

from src import config
from src.labels import build_labels, train_test_split
from src.tabular import (DEFAULT_GROUPS, NO_RECENCY_GROUPS, build_features, fit_model,
                         load_inputs, predict)

## Data, labels and features

In [ ]:
restaurants, reviews = load_inputs()
review_dates = reviews[["business_id", "date"]]
train, test = train_test_split(restaurants=restaurants, reviews=review_dates)
X_train = build_features(train, restaurants, reviews)
X_test = build_features(test, restaurants, reviews)

# Rows dropped under pre_cutoff_closures="exclude", to fit and check the same models on that label
def pre_cutoff_ids(cutoff):
    _, pre, _ = build_labels(cutoff, restaurants=restaurants, reviews=review_dates, return_details=True)
    return set(pre["business_id"])
keep = {"closed": (np.ones(len(train), bool), np.ones(len(test), bool)),
        "exclude": (~train["business_id"].isin(pre_cutoff_ids(config.TRAIN_CUTOFF)).to_numpy(),
                    ~test["business_id"].isin(pre_cutoff_ids(config.CUTOFF)).to_numpy())}
for label, (a, b) in keep.items():
    print(f"{label:8s} train {a.sum():,} rows, {train.closed[a].mean():.1%} closed | "
          f"test {b.sum():,} rows, {test.closed[b].mean():.1%} closed")

## Feature sets

- **Recency only**: days since the last review, as a single feature.
- **No recency**: drops days since last review and the 3- and 6-month counts, which mostly
  flag restaurants that went quiet (and may have closed) before the cutoff.
- **+ attributes**: sensitivity check only; these fields leak (see below).

In [ ]:
MODELS = [("Logistic regression", DEFAULT_GROUPS, "logit"),
          ("Gradient boosting, no recency", NO_RECENCY_GROUPS, "hgb"),
          ("Gradient boosting (headline)", DEFAULT_GROUPS, "hgb"),
          ("Gradient boosting + attributes (leaks)", DEFAULT_GROUPS + ["attributes"], "hgb")]
rows, fitted = [], {}
for label, (a, b) in keep.items():
    Xa, Xb, ya, yb = X_train[a], X_test[b], train.closed[a], test.closed[b]
    rows.append({"label": label, "model": "Recency only",
                 "avg_precision": average_precision_score(yb, Xb["days_since_last"])})
    for name, groups, kind in MODELS:
        model = fit_model(Xa, ya, groups=groups, kind=kind)
        fitted[label, name] = model
        rows.append({"label": label, "model": name,
                     "avg_precision": average_precision_score(yb, predict(model, Xb))})
pd.DataFrame(rows).pivot(index="model", columns="label", values="avg_precision").round(3)

## Why business attributes are excluded

The attributes are 2022 values. Survivors updated them after 2020 (e.g. adding delivery);
closed restaurants kept their last listing. So they mostly encode "still open in 2022".

In [ ]:
leak = pd.DataFrame({
    "Delivery": X_train["attr_RestaurantsDelivery"].map({1: "True", 0: "False"}).fillna("missing"),
    "More than 20 attributes": np.where(X_train["n_attributes"] > 20, "yes", "no"),
})
for col in leak:
    print(train.closed.groupby(leak[col]).agg(closure_rate="mean", restaurants="size").round(3), "\n")

## What drives the headline model (drop in test average precision when a feature is shuffled)

In [ ]:
model = fitted["closed", "Gradient boosting (headline)"]
cols = model.feature_columns_
pi = permutation_importance(model, X_test[cols], test.closed, scoring="average_precision",
                            n_repeats=5, random_state=config.RANDOM_SEED)
pd.Series(pi.importances_mean, cols).sort_values(ascending=False).head(15).round(4)

## Tuning

A 5-fold CV grid on the training year only (learning rate 0.03 to 0.1, 15 to 31 leaves,
minimum leaf 50 to 200) moved CV average precision by less than 0.01, so the defaults in
`HGB_PARAMS` are kept. Price range alone added nothing.